In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [4]:
from langchain_core.documents import Document
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [5]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [7]:
vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [8]:
vector_store.add_documents(docs)

['37cc9a99-38f6-4f70-85f8-9606b76201aa',
 '1387310a-9a6c-46d8-b207-ecd87314c102',
 'cd684842-d67a-4f74-b77b-9560b7e1fb61',
 '5c44ffd6-c8cc-48f7-b058-1fce57a88c2e',
 'b5ebb8de-74b5-4747-bf9c-f5820ca8235f']

In [9]:
vector_store.get(include=['embeddings','documents', 'metadatas'])


{'ids': ['37cc9a99-38f6-4f70-85f8-9606b76201aa',
  '1387310a-9a6c-46d8-b207-ecd87314c102',
  'cd684842-d67a-4f74-b77b-9560b7e1fb61',
  '5c44ffd6-c8cc-48f7-b058-1fce57a88c2e',
  'b5ebb8de-74b5-4747-bf9c-f5820ca8235f'],
 'embeddings': array([[ 0.00994726,  0.06914335, -0.05147114, ..., -0.03543343,
          0.01284807,  0.01248289],
        [ 0.00127747,  0.0312985 , -0.02375383, ..., -0.00518365,
         -0.03280613,  0.02737718],
        [-0.10265916,  0.02650811,  0.02271502, ..., -0.03359752,
         -0.07984938, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494367, ..., -0.10995808,
          0.00572556,  0.09915379],
        [ 0.01873977,  0.04382845, -0.04304255, ..., -0.07801621,
         -0.07840683, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [10]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='5c44ffd6-c8cc-48f7-b058-1fce57a88c2e', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='1387310a-9a6c-46d8-b207-ecd87314c102', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [ ]:
# search with similarity score
'''
diff between similarity_search and similarity_search_with_scoreThe core difference is that similarity_search only returns a list of matching documents, 
while similarity_search_with_score returns both the documents and their numerical similarity scores.
'''
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='5c44ffd6-c8cc-48f7-b058-1fce57a88c2e', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693601131439209),
 (Document(id='1387310a-9a6c-46d8-b207-ecd87314c102', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.14934504032135)]

In [ ]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='cd684842-d67a-4f74-b77b-9560b7e1fb61', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436002731323242),
 (Document(id='b5ebb8de-74b5-4747-bf9c-f5820ca8235f', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.8909374475479126)]

In [13]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [14]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['37cc9a99-38f6-4f70-85f8-9606b76201aa',
  '1387310a-9a6c-46d8-b207-ecd87314c102',
  'cd684842-d67a-4f74-b77b-9560b7e1fb61',
  '5c44ffd6-c8cc-48f7-b058-1fce57a88c2e',
  'b5ebb8de-74b5-4747-bf9c-f5820ca8235f'],
 'embeddings': array([[ 0.00994726,  0.06914335, -0.05147114, ..., -0.03543343,
          0.01284807,  0.01248289],
        [ 0.00127747,  0.0312985 , -0.02375383, ..., -0.00518365,
         -0.03280613,  0.02737718],
        [-0.10265916,  0.02650811,  0.02271502, ..., -0.03359752,
         -0.07984938, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494367, ..., -0.10995808,
          0.00572556,  0.09915379],
        [ 0.01873977,  0.04382845, -0.04304255, ..., -0.07801621,
         -0.07840683, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [15]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [16]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['37cc9a99-38f6-4f70-85f8-9606b76201aa',
  '1387310a-9a6c-46d8-b207-ecd87314c102',
  'cd684842-d67a-4f74-b77b-9560b7e1fb61',
  '5c44ffd6-c8cc-48f7-b058-1fce57a88c2e',
  'b5ebb8de-74b5-4747-bf9c-f5820ca8235f'],
 'embeddings': array([[ 0.00994726,  0.06914335, -0.05147114, ..., -0.03543343,
          0.01284807,  0.01248289],
        [ 0.00127747,  0.0312985 , -0.02375383, ..., -0.00518365,
         -0.03280613,  0.02737718],
        [-0.10265916,  0.02650811,  0.02271502, ..., -0.03359752,
         -0.07984938, -0.01507709],
        [ 0.02123396, -0.02468546, -0.04494367, ..., -0.10995808,
          0.00572556,  0.09915379],
        [ 0.01873977,  0.04382845, -0.04304255, ..., -0.07801621,
         -0.07840683, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo